# Local alternative feature summary

This notebook demonstrates `plotly.local.alternative_feature_summary` for one local alternative explanation. The plot is a local summary: it shows how often each feature participates in the alternatives for one explained instance. It is not global feature importance.

The main stacked bars show primary role plus quality-flag combinations such as `counter + ensured`, `counter + pareto`, and `counter + ensured + pareto`. `ensured` and `pareto` are quality flags represented inside the role bars, not a separate default status panel.

The optional conjunction panel is disabled by default. When enabled, it counts how often a feature participates in multi-feature rules. Unknown roles mean the role metadata was unavailable or unmapped, not that the rule has no role.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import subprocess
import sys
from pathlib import Path

package_dir = Path.cwd().resolve()
if package_dir.name == 'examples':
    package_dir = package_dir.parent
else:
    repo_candidate = Path('packages/visualization/calibrated-explanations-visualization-plotly').resolve()
    if repo_candidate.exists():
        package_dir = repo_candidate

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(package_dir)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly>=5.18'])

In [ ]:
import importlib

import calibrated_explanations.plugins.registry as registry
import ce_visualization_plotly.alternative_feature_summary as alternative_feature_summary_module
import ce_visualization_plotly.plugin as plotly_plugin_module
import numpy as np
from calibrated_explanations import WrapCalibratedExplainer
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

reset_catalog = getattr(registry, 'reset_plugin_catalog', None)
if callable(reset_catalog):
    reset_catalog(kind='all')
clear_env_cache = getattr(registry, 'clear_env_trust_cache', None)
if callable(clear_env_cache):
    clear_env_cache()
clear_warnings = getattr(registry, 'clear_trust_warnings', None)
if callable(clear_warnings):
    clear_warnings()
for module_name in [name for name in list(sys.modules) if name.startswith('ce_visualization_plotly')]:
    sys.modules.pop(module_name, None)
importlib.reload(alternative_feature_summary_module)
importlib.reload(plotly_plugin_module)
plotly_plugin_module.register_plotly_visualization_components()
np.set_printoptions(precision=3, suppress=True)

In [ ]:
X, y = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=5,
    n_redundant=0,
    random_state=0,
)

x_proper, x_holdout, y_proper, y_holdout = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=0,
    stratify=y,
)
x_cal, X_query, y_cal, y_query = train_test_split(
    x_holdout,
    y_holdout,
    test_size=0.5,
    random_state=0,
    stratify=y_holdout,
)

assert len(x_proper) == 300
assert len(x_cal) == 100
assert len(X_query) == 100

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=0)
explainer = WrapCalibratedExplainer(model)
explainer.fit(x_proper, y_proper)
assert explainer.fitted is True

explainer.calibrate(x_cal, y_cal)
assert explainer.calibrated is True

In [ ]:
alternatives = explainer.explore_alternatives(X_query[:5])

Default view: role-quality combinations only.

In [ ]:
alt = alternatives[0].plot(style="plotly.local.alternative_feature_summary", show=True)

Limit the display to the most involved features.

In [ ]:
alt = alternatives[0].plot(
    style="plotly.local.alternative_feature_summary",
    show=True,
    filter_top_features=5,
)

Normalize each feature row to shares while preserving raw counts in hover.

In [ ]:
alt = alternatives[0].plot(
    style="plotly.local.alternative_feature_summary",
    show=True,
    normalize="share",
)

Enable the optional conjunction panel. Conjunction bars count how often a feature appears in multi-feature rules.

In [ ]:
alternatives.add_conjunctions(max_rule_size=7)

In [ ]:
alt = alternatives[0].plot(
    style="plotly.local.alternative_feature_summary",
    show=True,
)

In [ ]:
alt = alternatives[0].plot(
    style="plotly.local.alternative_feature_summary",
    show=True,
    include_conjunctions=True,
)

Export to HTML without displaying the figure.

In [ ]:
alt = alternatives[0].plot(
    style="plotly.local.alternative_feature_summary",
    show=False,
    path="alternative_feature_summary.html",
)